# U-Net: Convolutional Networks for Biomedical Image Segmentation
## BME 562 / BME 506 — Medical Image Processing (Midterm Project)

| | |
|---|---|
| **Student** | Özgür Efe Zurnacı |
| **Student ID** | 221402001 |
| **Major** | Biomedical Engineering (BME) |
| **Instructor** | Seda Nilgün Dumlu, Ph.D. |
| **Semester** | Spring 2025–2026 |

**Paper:** Ronneberger, O., Fischer, P., & Brox, T. (2015). *U-Net: Convolutional Networks for Biomedical Image Segmentation.* [arXiv:1505.04597](https://arxiv.org/abs/1505.04597)

**Dataset:** [ISBI 2012 EM Segmentation Challenge](http://brainiac2.mit.edu/isbi_challenge/)

---

## 1. Introduction & Paper Summary

### 1.1 Problem Statement

Biomedical image segmentation is a critical task in medical image processing that requires assigning a class label to **every pixel** in an image, unlike standard classification which assigns a single label to the whole image. The key challenges are:

1. **Limited training data** — annotated biomedical images are scarce (often fewer than 50 images)
2. **Precise localization required** — segmentation must be pixel-accurate to be clinically useful
3. **Trade-off between context and localization** — larger input patches provide more context but lose spatial precision

Prior to U-Net, the best approach was the sliding-window CNN by Ciresan et al. (2012), which was slow (redundant patch-wise computation) and suffered from the localization/context trade-off.

### 1.2 U-Net Contribution

The U-Net architecture (Ronneberger et al., 2015) introduced an **encoder-decoder network with skip connections** that solves both challenges simultaneously:

- **Encoder (contracting path):** Captures hierarchical context through repeated convolutions and max pooling, following the structure of a typical CNN
- **Decoder (expansive path):** Recovers spatial resolution through up-convolutions (transposed convolutions), enabling precise localization
- **Skip connections:** High-resolution feature maps from the encoder are **concatenated** with the upsampled decoder features, allowing the network to combine coarse semantic information with fine spatial details
- **Data augmentation:** Extensive elastic deformations simulate realistic biological tissue variations, enabling the network to learn from very few annotated samples (as few as 30 images)

The resulting U-shaped architecture has 23 convolutional layers with no fully connected layers, giving it the ability to accept arbitrarily sized inputs.

### 1.3 Paper Results Summary

The U-Net was evaluated on three segmentation tasks:

| Challenge | Dataset | Metric | U-Net Score | Previous Best |
|---|---|---|---|---|
| ISBI 2012 EM Segmentation | Drosophila VNC (30 images, 512×512) | Warping Error | **0.000353** | 0.000420 |
| ISBI 2012 EM Segmentation | Drosophila VNC | Rand Error | **0.0382** | 0.0504 |
| ISBI 2015 Cell Tracking | PhC-U373 (Glioblastoma) | IoU | **0.9203** | 0.83 |
| ISBI 2015 Cell Tracking | DIC-HeLa | IoU | **0.7756** | 0.46 |

> **Key insight:** U-Net achieved state-of-the-art results with only 30 training images, demonstrating that architecture design + data augmentation can compensate for limited data.

---

## 2. Environment Setup & Data Loading

### 2.1 Library Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from glob import glob
import warnings
warnings.filterwarnings('ignore')

# Scikit-image for spatial and frequency domain processing
from skimage import filters, feature, exposure, morphology
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

# Scipy for FFT and advanced filtering
from scipy import ndimage, fft as scipy_fft
from scipy.ndimage import gaussian_filter, median_filter, uniform_filter

# OpenCV for additional image processing
import cv2

# Set plotting style
sns.set_theme(style="darkgrid", palette="deep")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

# Reproducibility
np.random.seed(42)

print("All libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"OpenCV version: {cv2.__version__}")

### 2.2 Data Loading

The ISBI 2012 EM Segmentation Challenge dataset consists of:
- **30 training images** (512×512 pixels, 8-bit grayscale) from serial section transmission electron microscopy (TEM) of *Drosophila* first instar larva ventral nerve cord (VNC)
- **30 corresponding ground truth segmentation maps** (binary: cell = white/255, membrane = black/0)
- **30 test images** (labels held by challenge organizers)

In [ ]:
# Define data paths
DATA_DIR = os.path.join('..', 'data', 'raw')
TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'train_images')
TRAIN_LBL_DIR = os.path.join(DATA_DIR, 'train_labels')
TEST_IMG_DIR = os.path.join(DATA_DIR, 'test_images')
FIGURES_DIR = os.path.join('..', 'figures')

# Create figures directory if it doesn't exist
os.makedirs(FIGURES_DIR, exist_ok=True)

# Load training images and labels
train_image_paths = sorted(glob(os.path.join(TRAIN_IMG_DIR, '*.png')))
train_label_paths = sorted(glob(os.path.join(TRAIN_LBL_DIR, '*.png')))
test_image_paths = sorted(glob(os.path.join(TEST_IMG_DIR, '*.png')))

# Load all images into numpy arrays
train_images = np.array([np.array(Image.open(p).convert('L')) for p in train_image_paths])
train_labels = np.array([np.array(Image.open(p).convert('L')) for p in train_label_paths])
test_images = np.array([np.array(Image.open(p).convert('L')) for p in test_image_paths])

print(f"Training images shape: {train_images.shape}  |  dtype: {train_images.dtype}")
print(f"Training labels shape: {train_labels.shape}  |  dtype: {train_labels.dtype}")
print(f"Test images shape:     {test_images.shape}  |  dtype: {test_images.dtype}")
print(f"\nPixel value range (images): [{train_images.min()}, {train_images.max()}]")
print(f"Pixel value range (labels): [{train_labels.min()}, {train_labels.max()}]")
print(f"\nUnique label values: {np.unique(train_labels)}")

### 2.3 Data Inspection — Sample Visualization

In [ ]:
# Display first 8 training images with their corresponding labels
fig, axes = plt.subplots(2, 8, figsize=(20, 5.5))
fig.suptitle('ISBI 2012 EM Segmentation Dataset — Training Samples', fontsize=15, fontweight='bold', y=1.02)

for i in range(8):
    axes[0, i].imshow(train_images[i], cmap='gray')
    axes[0, i].set_title(f'Image {i+1}', fontsize=10)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(train_labels[i], cmap='gray')
    axes[1, i].set_title(f'Label {i+1}', fontsize=10)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('EM Images', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Ground Truth', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'sample_images.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to figures/sample_images.png")

## 3. Exploratory Data Analysis (EDA)

### 3.1 Pixel Intensity Statistics

We compute descriptive statistics for both the EM images and the ground truth labels to understand the data distribution.

In [ ]:
# Compute per-image statistics
stats_data = []
for i in range(len(train_images)):
    img = train_images[i]
    lbl = train_labels[i]
    stats_data.append({
        'Image': f'Frame {i+1:02d}',
        'Img Mean': img.mean(),
        'Img Std': img.std(),
        'Img Median': np.median(img),
        'Img Min': img.min(),
        'Img Max': img.max(),
        'Label Mean': lbl.mean(),
        'Cell %': (lbl == 255).sum() / lbl.size * 100,
        'Membrane %': (lbl == 0).sum() / lbl.size * 100,
    })

stats_df = pd.DataFrame(stats_data)

# Display summary statistics
print("=" * 70)
print("TRAINING IMAGES — Pixel Intensity Statistics")
print("=" * 70)
print(f"{'Metric':<20} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 70)
print(f"{'Pixel Mean':<20} {stats_df['Img Mean'].mean():>10.2f} {stats_df['Img Mean'].std():>10.2f} {stats_df['Img Mean'].min():>10.2f} {stats_df['Img Mean'].max():>10.2f}")
print(f"{'Pixel Std':<20} {stats_df['Img Std'].mean():>10.2f} {stats_df['Img Std'].std():>10.2f} {stats_df['Img Std'].min():>10.2f} {stats_df['Img Std'].max():>10.2f}")
print(f"{'Pixel Median':<20} {stats_df['Img Median'].mean():>10.2f} {stats_df['Img Median'].std():>10.2f} {stats_df['Img Median'].min():>10.2f} {stats_df['Img Median'].max():>10.2f}")
print()
print("=" * 70)
print("CLASS DISTRIBUTION (Ground Truth Labels)")
print("=" * 70)
print(f"Average Cell (white) percentage:     {stats_df['Cell %'].mean():.2f}% ± {stats_df['Cell %'].std():.2f}%")
print(f"Average Membrane (black) percentage: {stats_df['Membrane %'].mean():.2f}% ± {stats_df['Membrane %'].std():.2f}%")
print()
stats_df.round(2)

### 3.2 Pixel Intensity Histograms

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Overall histogram of all training images
all_pixels = train_images.flatten()
axes[0].hist(all_pixels, bins=256, range=(0, 255), color='steelblue', alpha=0.8, edgecolor='none')
axes[0].set_title('(a) Pixel Intensity Distribution\n(All Training Images)', fontweight='bold')
axes[0].set_xlabel('Pixel Intensity')
axes[0].set_ylabel('Frequency')
axes[0].axvline(all_pixels.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean = {all_pixels.mean():.1f}')
axes[0].legend()

# (b) Per-image mean intensity distribution
axes[1].bar(range(1, 31), stats_df['Img Mean'], color='steelblue', alpha=0.8, edgecolor='navy', linewidth=0.5)
axes[1].axhline(stats_df['Img Mean'].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Overall Mean = {stats_df["Img Mean"].mean():.1f}')
axes[1].fill_between(range(0, 32),
                     stats_df['Img Mean'].mean() - stats_df['Img Mean'].std(),
                     stats_df['Img Mean'].mean() + stats_df['Img Mean'].std(),
                     alpha=0.15, color='red', label=f'± 1 Std = {stats_df["Img Mean"].std():.1f}')
axes[1].set_title('(b) Per-Image Mean Intensity', fontweight='bold')
axes[1].set_xlabel('Image Index')
axes[1].set_ylabel('Mean Pixel Intensity')
axes[1].set_xlim(0, 31)
axes[1].legend()

# (c) Class distribution
cell_pct = stats_df['Cell %'].values
membrane_pct = stats_df['Membrane %'].values
x = np.arange(1, 31)
axes[2].bar(x, cell_pct, color='#4CAF50', alpha=0.8, label='Cell (white)', edgecolor='none')
axes[2].bar(x, membrane_pct, bottom=cell_pct, color='#37474F', alpha=0.8, label='Membrane (black)', edgecolor='none')
axes[2].set_title('(c) Class Distribution per Image', fontweight='bold')
axes[2].set_xlabel('Image Index')
axes[2].set_ylabel('Percentage (%)')
axes[2].legend()
axes[2].set_xlim(0, 31)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_histograms.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to figures/eda_histograms.png")

### 3.3 Ground Truth Overlay Visualization

Overlaying the segmentation mask on the original EM images helps us understand the relationship between the image intensities and the membrane structures.

In [ ]:
# Show overlays of ground truth on original images
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('Ground Truth Overlay — Membrane Boundaries on EM Images', fontsize=15, fontweight='bold', y=1.01)

sample_indices = [0, 5, 10, 15, 20, 24, 28, 29]

for idx, ax in zip(sample_indices, axes.flatten()):
    img = train_images[idx]
    lbl = train_labels[idx]
    
    # Create RGB overlay
    overlay = np.stack([img, img, img], axis=-1).astype(np.float32) / 255.0
    
    # Highlight membranes (label==0) in red
    membrane_mask = (lbl == 0)
    overlay[membrane_mask, 0] = 1.0   # Red channel
    overlay[membrane_mask, 1] *= 0.3  # Dim green
    overlay[membrane_mask, 2] *= 0.3  # Dim blue
    
    ax.imshow(overlay)
    cell_ratio = (lbl == 255).sum() / lbl.size * 100
    ax.set_title(f'Frame {idx+1} — Cell: {cell_ratio:.1f}%', fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'ground_truth_overlay.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to figures/ground_truth_overlay.png")

### 3.4 EDA Summary

**Key findings from EDA:**

1. **Image characteristics:** The EM images are 512×512 8-bit grayscale with a moderate average intensity and broad pixel distribution, reflecting the diverse subcellular structures visible in electron microscopy.

2. **Class imbalance:** There is a notable imbalance between cell (majority) and membrane (minority) classes. This imbalance must be considered during model training, and techniques like weighted loss functions (as proposed in the U-Net paper) are important.

3. **Membrane structure:** The membranes form thin, connected boundaries between cells. Accurate segmentation requires the model to detect these fine structures — this is where the U-Net's skip connections and high-resolution features become critical.

4. **Inter-image consistency:** The per-image mean intensities show moderate variation across the 30 training images, indicating that the dataset captures different tissue regions with varying contrast.

---

## 4. Spatial-Domain Processing (Weeks 1–5) ⭐

This section demonstrates image processing techniques in the **spatial domain**, as covered in Weeks 1–5 of the BME 562/506 course. We apply smoothing filters, sharpening, and edge detection to the EM images and analyze their effectiveness for membrane structure detection.

### 4.1 Smoothing Filters

Smoothing (low-pass) filters reduce noise by averaging pixel values with their neighbors. We compare three common smoothing approaches:

1. **Gaussian filter** — weighted average with a Gaussian kernel (σ controls smoothing strength)
2. **Median filter** — replaces each pixel with the median of its neighborhood (effective for salt-and-pepper noise)
3. **Bilateral filter** — edge-preserving smoothing that considers both spatial and intensity distances

In [ ]:
# Select a representative image for spatial processing demonstrations
sample_idx = 0
sample_img = train_images[sample_idx].astype(np.float64) / 255.0
sample_lbl = train_labels[sample_idx]

print(f"Working with: Frame {sample_idx + 1} (512×512, normalized to [0, 1])")
print(f"Pixel range: [{sample_img.min():.3f}, {sample_img.max():.3f}]")

In [ ]:
# ---- 4.1.1 Gaussian Smoothing ----
sigmas = [1, 2, 3, 5]

fig, axes = plt.subplots(2, len(sigmas) + 1, figsize=(20, 8))
fig.suptitle('4.1.1 — Gaussian Smoothing at Different Scales (σ)', fontsize=15, fontweight='bold', y=1.02)

# Top row: smoothed images
axes[0, 0].imshow(sample_img, cmap='gray', vmin=0, vmax=1)
axes[0, 0].set_title('Original', fontweight='bold')
axes[0, 0].axis('off')

# Bottom row: difference (what was removed = noise/detail)
axes[1, 0].imshow(np.zeros_like(sample_img), cmap='RdBu_r', vmin=-0.3, vmax=0.3)
axes[1, 0].set_title('Difference: N/A', fontweight='bold')
axes[1, 0].axis('off')

for i, sigma in enumerate(sigmas, 1):
    smoothed = gaussian_filter(sample_img, sigma=sigma)
    diff = sample_img - smoothed
    
    axes[0, i].imshow(smoothed, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'σ = {sigma}', fontweight='bold')
    axes[0, i].axis('off')
    
    im = axes[1, i].imshow(diff, cmap='RdBu_r', vmin=-0.3, vmax=0.3)
    axes[1, i].set_title(f'Removed Detail (σ={sigma})', fontweight='bold')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Smoothed', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Difference', fontsize=12, fontweight='bold')

plt.colorbar(im, ax=axes[1, :], shrink=0.6, label='Intensity Difference')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'spatial_gaussian_smoothing.png'), dpi=150, bbox_inches='tight')
plt.show()
print("As σ increases, more fine details (including membrane boundaries) are suppressed.")

In [ ]:
# ---- 4.1.2 Median Filter ----
kernel_sizes = [3, 5, 7, 9]

fig, axes = plt.subplots(2, len(kernel_sizes) + 1, figsize=(20, 8))
fig.suptitle('4.1.2 — Median Filter at Different Kernel Sizes', fontsize=15, fontweight='bold', y=1.02)

axes[0, 0].imshow(sample_img, cmap='gray', vmin=0, vmax=1)
axes[0, 0].set_title('Original', fontweight='bold')
axes[0, 0].axis('off')

axes[1, 0].imshow(np.zeros_like(sample_img), cmap='RdBu_r', vmin=-0.3, vmax=0.3)
axes[1, 0].set_title('Difference: N/A', fontweight='bold')
axes[1, 0].axis('off')

for i, ks in enumerate(kernel_sizes, 1):
    med = median_filter(sample_img, size=ks)
    diff = sample_img - med
    
    axes[0, i].imshow(med, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'Kernel = {ks}×{ks}', fontweight='bold')
    axes[0, i].axis('off')
    
    im = axes[1, i].imshow(diff, cmap='RdBu_r', vmin=-0.3, vmax=0.3)
    axes[1, i].set_title(f'Removed Detail ({ks}×{ks})', fontweight='bold')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Filtered', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Difference', fontsize=12, fontweight='bold')

plt.colorbar(im, ax=axes[1, :], shrink=0.6, label='Intensity Difference')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'spatial_median_filter.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Median filter preserves edges better than Gaussian while removing impulse noise.")

In [ ]:
# ---- 4.1.3 Bilateral Filter (Edge-Preserving Smoothing) ----
# OpenCV bilateral filter: d=diameter, sigmaColor, sigmaSpace
sample_img_uint8 = (sample_img * 255).astype(np.uint8)

bilateral_params = [(5, 25, 25), (9, 50, 50), (9, 75, 75), (15, 100, 100)]
param_labels = ['d=5, σ=25', 'd=9, σ=50', 'd=9, σ=75', 'd=15, σ=100']

fig, axes = plt.subplots(2, len(bilateral_params) + 1, figsize=(20, 8))
fig.suptitle('4.1.3 — Bilateral Filter (Edge-Preserving Smoothing)', fontsize=15, fontweight='bold', y=1.02)

axes[0, 0].imshow(sample_img, cmap='gray', vmin=0, vmax=1)
axes[0, 0].set_title('Original', fontweight='bold')
axes[0, 0].axis('off')

axes[1, 0].imshow(np.zeros_like(sample_img), cmap='RdBu_r', vmin=-0.3, vmax=0.3)
axes[1, 0].set_title('Difference: N/A', fontweight='bold')
axes[1, 0].axis('off')

for i, ((d, sc, ss), label) in enumerate(zip(bilateral_params, param_labels), 1):
    bilateral = cv2.bilateralFilter(sample_img_uint8, d, sc, ss).astype(np.float64) / 255.0
    diff = sample_img - bilateral
    
    axes[0, i].imshow(bilateral, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(label, fontweight='bold')
    axes[0, i].axis('off')
    
    im = axes[1, i].imshow(diff, cmap='RdBu_r', vmin=-0.3, vmax=0.3)
    axes[1, i].set_title(f'Removed Detail', fontweight='bold')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Filtered', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Difference', fontsize=12, fontweight='bold')

plt.colorbar(im, ax=axes[1, :], shrink=0.6, label='Intensity Difference')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'spatial_bilateral_filter.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Bilateral filter smooths homogeneous regions while preserving edges — ideal for denoising before segmentation.")

### 4.1.4 Quantitative Comparison of Smoothing Filters

We compare smoothing filters using **PSNR** (Peak Signal-to-Noise Ratio) and **SSIM** (Structural Similarity Index) relative to the original image. Higher SSIM means the filter preserves more structural information.

In [ ]:
# Quantitative comparison of all smoothing filters
filter_results = []

# Test on multiple images
test_indices = [0, 5, 10, 15, 20, 25, 29]

for idx in test_indices:
    img = train_images[idx].astype(np.float64) / 255.0
    img_uint8 = train_images[idx]
    
    # Gaussian filters
    for sigma in [1, 2, 3]:
        smoothed = gaussian_filter(img, sigma=sigma)
        filter_results.append({
            'Filter': f'Gaussian (σ={sigma})',
            'Type': 'Gaussian',
            'Image': idx + 1,
            'PSNR': psnr(img, smoothed, data_range=1.0),
            'SSIM': ssim(img, smoothed, data_range=1.0)
        })
    
    # Median filters
    for ks in [3, 5, 7]:
        med = median_filter(img, size=ks)
        filter_results.append({
            'Filter': f'Median ({ks}×{ks})',
            'Type': 'Median',
            'Image': idx + 1,
            'PSNR': psnr(img, med, data_range=1.0),
            'SSIM': ssim(img, med, data_range=1.0)
        })
    
    # Bilateral filters
    for d, sc, ss, label in [(5, 25, 25, 'd=5,σ=25'), (9, 50, 50, 'd=9,σ=50'), (9, 75, 75, 'd=9,σ=75')]:
        bilateral = cv2.bilateralFilter(img_uint8, d, sc, ss).astype(np.float64) / 255.0
        filter_results.append({
            'Filter': f'Bilateral ({label})',
            'Type': 'Bilateral',
            'Image': idx + 1,
            'PSNR': psnr(img, bilateral, data_range=1.0),
            'SSIM': ssim(img, bilateral, data_range=1.0)
        })

results_df = pd.DataFrame(filter_results)

# Summary table
summary = results_df.groupby('Filter').agg(
    PSNR_Mean=('PSNR', 'mean'),
    PSNR_Std=('PSNR', 'std'),
    SSIM_Mean=('SSIM', 'mean'),
    SSIM_Std=('SSIM', 'std')
).round(4).sort_values('SSIM_Mean', ascending=False)

print("=" * 75)
print("SMOOTHING FILTER COMPARISON (Averaged over 7 test images)")
print("=" * 75)
print(f"{'Filter':<25} {'PSNR (dB)':>15} {'SSIM':>15}")
print("-" * 75)
for name, row in summary.iterrows():
    print(f"{name:<25} {row['PSNR_Mean']:>8.2f} ± {row['PSNR_Std']:>4.2f}   {row['SSIM_Mean']:>7.4f} ± {row['SSIM_Std']:>6.4f}")

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Smoothing Filter Comparison — PSNR & SSIM', fontsize=14, fontweight='bold')

colors = {'Gaussian': '#2196F3', 'Median': '#4CAF50', 'Bilateral': '#FF9800'}
filter_colors = [colors[results_df[results_df['Filter']==f]['Type'].iloc[0]] for f in summary.index]

axes[0].barh(range(len(summary)), summary['PSNR_Mean'], xerr=summary['PSNR_Std'],
             color=filter_colors, alpha=0.85, edgecolor='gray', linewidth=0.5)
axes[0].set_yticks(range(len(summary)))
axes[0].set_yticklabels(summary.index, fontsize=9)
axes[0].set_xlabel('PSNR (dB) — Higher is better')
axes[0].set_title('(a) PSNR', fontweight='bold')

axes[1].barh(range(len(summary)), summary['SSIM_Mean'], xerr=summary['SSIM_Std'],
             color=filter_colors, alpha=0.85, edgecolor='gray', linewidth=0.5)
axes[1].set_yticks(range(len(summary)))
axes[1].set_yticklabels(summary.index, fontsize=9)
axes[1].set_xlabel('SSIM — Higher is better')
axes[1].set_title('(b) SSIM', fontweight='bold')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=l) for l, c in colors.items()]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=10, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'smoothing_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

### 4.2 Edge Detection

Edge detection is fundamental for identifying membrane boundaries in EM images. We apply three classical edge detection methods and compare their outputs against the ground truth segmentation.

1. **Sobel filter** — first-order derivative, detects gradient magnitude (horizontal + vertical)
2. **Canny edge detector** — multi-stage: Gaussian → gradient → non-maximum suppression → hysteresis thresholding
3. **Laplacian of Gaussian (LoG)** — second-order derivative, detects zero-crossings at edges

In [ ]:
# ---- 4.2.1 Sobel Edge Detection ----

# Compute Sobel edges
sobel_x = filters.sobel_h(sample_img)  # Horizontal edges
sobel_y = filters.sobel_v(sample_img)  # Vertical edges
sobel_mag = filters.sobel(sample_img)  # Gradient magnitude

fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))
fig.suptitle('4.2.1 — Sobel Edge Detection', fontsize=15, fontweight='bold', y=1.05)

axes[0].imshow(sample_img, cmap='gray')
axes[0].set_title('Original', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(sobel_x, cmap='RdBu_r')
axes[1].set_title('Sobel Horizontal (∂/∂x)', fontweight='bold')
axes[1].axis('off')

axes[2].imshow(sobel_y, cmap='RdBu_r')
axes[2].set_title('Sobel Vertical (∂/∂y)', fontweight='bold')
axes[2].axis('off')

axes[3].imshow(sobel_mag, cmap='hot')
axes[3].set_title('Sobel Magnitude |∇I|', fontweight='bold')
axes[3].axis('off')

# Compare with ground truth membrane (inverted label: membrane=1)
membrane_gt = 1.0 - (sample_lbl.astype(np.float64) / 255.0)
axes[4].imshow(membrane_gt, cmap='hot')
axes[4].set_title('Ground Truth Membrane', fontweight='bold')
axes[4].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'edge_sobel.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Sobel gradient magnitude highlights edges that correspond to membrane locations in the ground truth.")

In [ ]:
# ---- 4.2.2 Canny Edge Detection ----
# Test multiple sigma values for Canny

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('4.2.2 — Canny Edge Detection at Different Scales', fontsize=15, fontweight='bold', y=1.02)

canny_sigmas = [0.5, 1.0, 2.0, 3.0]
sample_img_uint8_local = (sample_img * 255).astype(np.uint8)

for i, sigma in enumerate(canny_sigmas):
    # Scikit-image Canny
    edges = feature.canny(sample_img, sigma=sigma)
    
    axes[0, i].imshow(edges, cmap='gray')
    axes[0, i].set_title(f'Canny (σ={sigma})', fontweight='bold')
    axes[0, i].axis('off')
    
    # Overlay edges on original
    overlay = np.stack([sample_img, sample_img, sample_img], axis=-1)
    overlay[edges, 0] = 1.0  # Red edges
    overlay[edges, 1] = 0.0
    overlay[edges, 2] = 0.0
    
    axes[1, i].imshow(overlay)
    axes[1, i].set_title(f'Overlay (σ={sigma})', fontweight='bold')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Edge Map', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Overlay', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'edge_canny.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Canny with σ=1.0-2.0 provides the best balance between membrane detection and noise suppression.")

In [ ]:
# ---- 4.2.3 Laplacian of Gaussian (LoG) ----
fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))
fig.suptitle('4.2.3 — Laplacian of Gaussian (LoG) Edge Detection', fontsize=15, fontweight='bold', y=1.05)

axes[0].imshow(sample_img, cmap='gray')
axes[0].set_title('Original', fontweight='bold')
axes[0].axis('off')

log_sigmas = [1, 2, 3, 5]
for i, sigma in enumerate(log_sigmas):
    # Apply Gaussian then Laplacian
    log_img = ndimage.gaussian_laplace(sample_img, sigma=sigma)
    
    axes[i+1].imshow(log_img, cmap='RdBu_r')
    axes[i+1].set_title(f'LoG (σ={sigma})', fontweight='bold')
    axes[i+1].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'edge_log.png'), dpi=150, bbox_inches='tight')
plt.show()
print("LoG detects edges at different scales. σ=1-2 captures fine membrane details; σ=3-5 captures coarser structures.")

### 4.2.4 Quantitative Edge Detection Comparison with Ground Truth

We systematically compare each edge detection method against the ground truth membrane labels using the **F1-Score** (harmonic mean of Precision and Recall). The ground truth membranes are extracted by inverting and then skeletonizing the label maps.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate_edge_detection(edge_map, ground_truth_membrane, dilation_radius=2):
    """
    Evaluate edge detection against ground truth membranes.
    We dilate the ground truth slightly (2px) to allow for small offsets.
    """
    # Dilate ground truth to allow near-matches
    gt_dilated = morphology.binary_dilation(ground_truth_membrane, morphology.disk(dilation_radius))
    
    # Flatten
    pred = edge_map.flatten().astype(bool)
    gt = gt_dilated.flatten().astype(bool)
    
    prec = precision_score(gt, pred, zero_division=0)
    rec = recall_score(gt, pred, zero_division=0)
    f1 = f1_score(gt, pred, zero_division=0)
    
    return prec, rec, f1

# Evaluate on multiple images
edge_results = []
eval_indices = list(range(0, 30, 3))  # Every 3rd image

for idx in eval_indices:
    img = train_images[idx].astype(np.float64) / 255.0
    lbl = train_labels[idx]
    membrane_gt = (lbl < 128)  # Membrane = dark pixels in label
    
    # Sobel
    sobel = filters.sobel(img)
    sobel_binary = sobel > filters.threshold_otsu(sobel)
    p, r, f = evaluate_edge_detection(sobel_binary, membrane_gt)
    edge_results.append({'Method': 'Sobel + Otsu', 'Image': idx+1, 'Precision': p, 'Recall': r, 'F1': f})
    
    # Canny at different sigmas
    for sigma in [0.5, 1.0, 2.0, 3.0]:
        canny_edges = feature.canny(img, sigma=sigma)
        p, r, f = evaluate_edge_detection(canny_edges, membrane_gt)
        edge_results.append({'Method': f'Canny (σ={sigma})', 'Image': idx+1, 'Precision': p, 'Recall': r, 'F1': f})
    
    # LoG + zero-crossing approximation
    for sigma in [1, 2, 3]:
        log_img = ndimage.gaussian_laplace(img, sigma=sigma)
        # Zero-crossing detection (sign change)
        log_binary = np.abs(log_img) > np.percentile(np.abs(log_img), 85)
        p, r, f = evaluate_edge_detection(log_binary, membrane_gt)
        edge_results.append({'Method': f'LoG (σ={sigma})', 'Image': idx+1, 'Precision': p, 'Recall': r, 'F1': f})

edge_df = pd.DataFrame(edge_results)

# Summary
edge_summary = edge_df.groupby('Method').agg(
    Precision=('Precision', 'mean'),
    Recall=('Recall', 'mean'),
    F1=('F1', 'mean')
).round(4).sort_values('F1', ascending=False)

print("=" * 70)
print("EDGE DETECTION vs GROUND TRUTH MEMBRANE — F1 Score Comparison")
print("=" * 70)
print(f"{'Method':<20} {'Precision':>12} {'Recall':>12} {'F1-Score':>12}")
print("-" * 70)
for name, row in edge_summary.iterrows():
    print(f"{name:<20} {row['Precision']:>12.4f} {row['Recall']:>12.4f} {row['F1']:>12.4f}")

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
methods = edge_summary.index.tolist()
x = np.arange(len(methods))
width = 0.25

bars1 = ax.bar(x - width, edge_summary['Precision'], width, label='Precision', color='#2196F3', alpha=0.85)
bars2 = ax.bar(x, edge_summary['Recall'], width, label='Recall', color='#4CAF50', alpha=0.85)
bars3 = ax.bar(x + width, edge_summary['F1'], width, label='F1-Score', color='#FF5722', alpha=0.85)

ax.set_xlabel('Edge Detection Method')
ax.set_ylabel('Score')
ax.set_title('Edge Detection Methods vs. Ground Truth Membrane Labels', fontweight='bold', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=30, ha='right', fontsize=9)
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'edge_detection_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

### 4.3 Spatial-Domain Processing Summary

**Key findings:**

| Technique | Observation |
|---|---|
| **Gaussian smoothing** | Effective for noise reduction but progressively blurs membrane boundaries as σ increases. Suitable as preprocessing before edge detection. |
| **Median filter** | Better edge preservation than Gaussian; effective for impulse noise. The 3×3 kernel is optimal for this dataset. |
| **Bilateral filter** | Best edge-preserving smoothing — maintains sharp membrane boundaries while smoothing cell interiors. Ideal preprocessing for segmentation. |
| **Sobel edge detection** | Provides continuous edge maps with gradient magnitudes. When thresholded (Otsu), it captures membrane boundaries but also picks up intracellular structures. |
| **Canny edge detection** | Produces clean, thin edge maps. σ=1.0 provides the best F1-score against ground truth membranes for this dataset. |
| **LoG edge detection** | Multi-scale analysis capability. Lower σ values capture membrane details, higher σ values detect broader cell boundaries. |

> **Relevance to U-Net:** These spatial-domain techniques demonstrate the features that the early layers of U-Net's contracting path learn automatically — edge detection and smoothing are implicitly performed by the learned convolutional filters. Understanding these classical methods provides insight into what the neural network learns at different depths.

---

## 5. Frequency-Domain Processing (Week 6) ⭐

This section demonstrates image processing in the **frequency domain** using the 2D Fast Fourier Transform (FFT). The Fourier Transform decomposes an image into its constituent frequency components:
- **Low frequencies** → smooth, slowly varying regions (overall brightness, gradual intensity changes)
- **High frequencies** → rapid intensity changes (edges, fine details, noise)

### 5.1 2D FFT — Magnitude Spectrum Visualization

In [ ]:
# Compute and display the 2D FFT magnitude spectrum for sample images
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('5.1 — 2D FFT Magnitude Spectrum of EM Images', fontsize=15, fontweight='bold', y=1.02)

demo_indices = [0, 5, 15, 25]
for col, idx in enumerate(demo_indices):
    img = train_images[idx].astype(np.float64)

    # Compute 2D FFT
    f_transform = np.fft.fft2(img)
    f_shift = np.fft.fftshift(f_transform)  # Shift DC component to center
    magnitude_spectrum = np.log1p(np.abs(f_shift))  # Log scale for visibility

    axes[0, col].imshow(img, cmap='gray')
    axes[0, col].set_title(f'Frame {idx+1} — Spatial', fontweight='bold')
    axes[0, col].axis('off')

    axes[1, col].imshow(magnitude_spectrum, cmap='inferno')
    axes[1, col].set_title(f'Frame {idx+1} — FFT Magnitude', fontweight='bold')
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Spatial Domain', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Frequency Domain', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fft_magnitude_spectrum.png'), dpi=150, bbox_inches='tight')
plt.show()
print("The bright center represents low frequencies (smooth areas).")
print("Patterns radiating outward represent high frequencies (edges, textures).")

### 5.2 Frequency-Domain Filtering — Filter Design

We design three types of low-pass and high-pass filters in the frequency domain:
1. **Ideal filter** — sharp cutoff (causes ringing artifacts)
2. **Butterworth filter** — smooth rolloff (order n controls sharpness)
3. **Gaussian filter** — smoothest rolloff (no ringing)

In [ ]:
def create_frequency_filters(shape, cutoff, order=2):
    """Create Ideal, Butterworth, and Gaussian frequency-domain filters."""
    rows, cols = shape
    crow, ccol = rows // 2, cols // 2
    u = np.arange(rows).reshape(-1, 1) - crow
    v = np.arange(cols).reshape(1, -1) - ccol
    D = np.sqrt(u**2 + v**2)  # Distance from center

    # Ideal Low-Pass
    ideal_lp = (D <= cutoff).astype(np.float64)

    # Butterworth Low-Pass
    butterworth_lp = 1 / (1 + (D / cutoff) ** (2 * order))

    # Gaussian Low-Pass
    gaussian_lp = np.exp(-(D**2) / (2 * (cutoff**2)))

    return {
        'Ideal LP': ideal_lp,
        'Butterworth LP': butterworth_lp,
        'Gaussian LP': gaussian_lp,
        'Ideal HP': 1 - ideal_lp,
        'Butterworth HP': 1 - butterworth_lp,
        'Gaussian HP': 1 - gaussian_lp,
    }

# Visualize the filter shapes
cutoffs = [20, 40, 80]
fig, axes = plt.subplots(2, len(cutoffs), figsize=(18, 10))
fig.suptitle('Frequency-Domain Filter Profiles (1D Cross-Section)', fontsize=15, fontweight='bold')

for col, cutoff in enumerate(cutoffs):
    filters = create_frequency_filters((512, 512), cutoff)
    center = 256
    freq_axis = np.arange(512) - center

    # Low-pass profiles
    axes[0, col].plot(freq_axis, filters['Ideal LP'][center, :], 'r-', linewidth=2, label='Ideal')
    axes[0, col].plot(freq_axis, filters['Butterworth LP'][center, :], 'g-', linewidth=2, label='Butterworth (n=2)')
    axes[0, col].plot(freq_axis, filters['Gaussian LP'][center, :], 'b-', linewidth=2, label='Gaussian')
    axes[0, col].set_title(f'Low-Pass (D₀={cutoff})', fontweight='bold')
    axes[0, col].set_xlabel('Frequency')
    axes[0, col].set_ylabel('Filter Response')
    axes[0, col].legend(fontsize=8)
    axes[0, col].set_ylim(-0.05, 1.1)
    axes[0, col].grid(True, alpha=0.3)

    # High-pass profiles
    axes[1, col].plot(freq_axis, filters['Ideal HP'][center, :], 'r-', linewidth=2, label='Ideal')
    axes[1, col].plot(freq_axis, filters['Butterworth HP'][center, :], 'g-', linewidth=2, label='Butterworth (n=2)')
    axes[1, col].plot(freq_axis, filters['Gaussian HP'][center, :], 'b-', linewidth=2, label='Gaussian')
    axes[1, col].set_title(f'High-Pass (D₀={cutoff})', fontweight='bold')
    axes[1, col].set_xlabel('Frequency')
    axes[1, col].set_ylabel('Filter Response')
    axes[1, col].legend(fontsize=8)
    axes[1, col].set_ylim(-0.05, 1.1)
    axes[1, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'freq_filter_profiles.png'), dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Low-Pass Filtering — Smoothing in the Frequency Domain

In [ ]:
def apply_freq_filter(image, filt):
    """Apply a frequency-domain filter to an image."""
    f_transform = np.fft.fft2(image.astype(np.float64))
    f_shift = np.fft.fftshift(f_transform)
    filtered = f_shift * filt
    f_ishift = np.fft.ifftshift(filtered)
    result = np.real(np.fft.ifft2(f_ishift))
    return np.clip(result, 0, 255)

# Apply low-pass filters at different cutoff frequencies
sample = train_images[0].astype(np.float64)
cutoff_values = [20, 40, 80]
filter_types = ['Ideal LP', 'Butterworth LP', 'Gaussian LP']

fig, axes = plt.subplots(len(filter_types), len(cutoff_values) + 1, figsize=(20, 14))
fig.suptitle('5.3 — Low-Pass Filtering in Frequency Domain', fontsize=15, fontweight='bold', y=1.01)

for row, ftype in enumerate(filter_types):
    axes[row, 0].imshow(sample, cmap='gray', vmin=0, vmax=255)
    axes[row, 0].set_title('Original', fontweight='bold')
    axes[row, 0].axis('off')
    axes[row, 0].set_ylabel(ftype, fontsize=12, fontweight='bold')

    for col, cutoff in enumerate(cutoff_values, 1):
        filters = create_frequency_filters((512, 512), cutoff)
        result = apply_freq_filter(sample, filters[ftype])

        axes[row, col].imshow(result, cmap='gray', vmin=0, vmax=255)
        axes[row, col].set_title(f'D₀ = {cutoff}', fontweight='bold')
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'freq_lowpass.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Lower cutoff → more smoothing. Ideal filter shows ringing artifacts (Gibbs phenomenon).")
print("Gaussian LP provides the smoothest result with no ringing — equivalent to spatial Gaussian blur.")

### 5.4 High-Pass Filtering — Edge Enhancement in the Frequency Domain

In [ ]:
# Apply high-pass filters
filter_types_hp = ['Ideal HP', 'Butterworth HP', 'Gaussian HP']
cutoff_values_hp = [20, 40, 80]

fig, axes = plt.subplots(len(filter_types_hp), len(cutoff_values_hp) + 1, figsize=(20, 14))
fig.suptitle('5.4 — High-Pass Filtering in Frequency Domain', fontsize=15, fontweight='bold', y=1.01)

for row, ftype in enumerate(filter_types_hp):
    axes[row, 0].imshow(sample, cmap='gray', vmin=0, vmax=255)
    axes[row, 0].set_title('Original', fontweight='bold')
    axes[row, 0].axis('off')
    axes[row, 0].set_ylabel(ftype, fontsize=12, fontweight='bold')

    for col, cutoff in enumerate(cutoff_values_hp, 1):
        filters = create_frequency_filters((512, 512), cutoff)
        result = apply_freq_filter(sample, filters[ftype])

        axes[row, col].imshow(result, cmap='gray')
        axes[row, col].set_title(f'D₀ = {cutoff}', fontweight='bold')
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'freq_highpass.png'), dpi=150, bbox_inches='tight')
plt.show()
print("High-pass filtering extracts edges and fine details — compare with Sobel/Canny from Section 4.")
print("Higher cutoff → more detail retained; lower cutoff → only the strongest edges remain.")

### 5.5 Band-Pass Filtering — Isolating Specific Frequency Bands

In [ ]:
def create_bandpass_filter(shape, low_cutoff, high_cutoff):
    """Create a Gaussian band-pass filter."""
    rows, cols = shape
    crow, ccol = rows // 2, cols // 2
    u = np.arange(rows).reshape(-1, 1) - crow
    v = np.arange(cols).reshape(1, -1) - ccol
    D = np.sqrt(u**2 + v**2)
    lp_high = np.exp(-(D**2) / (2 * high_cutoff**2))
    lp_low = np.exp(-(D**2) / (2 * low_cutoff**2))
    return lp_high - lp_low

# Apply band-pass filters to isolate different frequency bands
bands = [(5, 20), (20, 50), (50, 100), (100, 200)]
band_labels = ['5-20 (Coarse)', '20-50 (Medium)', '50-100 (Fine)', '100-200 (V. Fine)']

fig, axes = plt.subplots(2, len(bands) + 1, figsize=(22, 9))
fig.suptitle('5.5 — Band-Pass Filtering: Isolating Frequency Bands', fontsize=15, fontweight='bold', y=1.02)

axes[0, 0].imshow(sample, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title('Original', fontweight='bold')
axes[0, 0].axis('off')

# Show the filter mask
axes[1, 0].imshow(np.zeros((512,512)), cmap='hot', vmin=0, vmax=1)
axes[1, 0].set_title('Filter: N/A', fontweight='bold')
axes[1, 0].axis('off')

for i, ((lo, hi), label) in enumerate(zip(bands, band_labels), 1):
    bp_filter = create_bandpass_filter((512, 512), lo, hi)
    result = apply_freq_filter(sample, bp_filter)

    axes[0, i].imshow(result, cmap='gray')
    axes[0, i].set_title(f'Band: {label}', fontweight='bold')
    axes[0, i].axis('off')

    axes[1, i].imshow(bp_filter, cmap='hot', vmin=0, vmax=1)
    axes[1, i].set_title(f'Filter Mask', fontweight='bold')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Filtered Image', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Filter Mask', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'freq_bandpass.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Band-pass filtering isolates structures at specific scales.")
print("The 20-50 band captures membrane-scale structures effectively.")

### 5.6 Spatial vs. Frequency Domain Comparison

In [ ]:
# Compare spatial Gaussian blur with frequency-domain Gaussian LP filter
from scipy.ndimage import gaussian_filter as sp_gaussian

sample_norm = sample / 255.0

# Spatial domain: sigma = 3 Gaussian blur
spatial_gauss = sp_gaussian(sample_norm, sigma=3)

# Frequency domain: equivalent Gaussian LP
# The relationship: frequency cutoff D0 ≈ (image_size) / (2*pi*sigma)
D0_equiv = 512 / (2 * np.pi * 3)  # ≈ 27
filters = create_frequency_filters((512, 512), D0_equiv)
freq_gauss = apply_freq_filter(sample, filters['Gaussian LP']) / 255.0

# Compute difference
diff = np.abs(spatial_gauss - freq_gauss)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('5.6 — Spatial vs. Frequency-Domain Gaussian Smoothing', fontsize=15, fontweight='bold', y=1.05)

axes[0].imshow(sample_norm, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Original', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(spatial_gauss, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Spatial Gaussian (σ=3)', fontweight='bold')
axes[1].axis('off')

axes[2].imshow(freq_gauss, cmap='gray', vmin=0, vmax=1)
axes[2].set_title(f'Frequency Gaussian LP (D₀≈{D0_equiv:.0f})', fontweight='bold')
axes[2].axis('off')

im = axes[3].imshow(diff, cmap='hot', vmin=0, vmax=0.05)
axes[3].set_title('Absolute Difference', fontweight='bold')
axes[3].axis('off')
plt.colorbar(im, ax=axes[3], shrink=0.8)

ssim_val = ssim(spatial_gauss, freq_gauss, data_range=1.0)
psnr_val = psnr(spatial_gauss, freq_gauss, data_range=1.0)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'spatial_vs_freq_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"SSIM between spatial and frequency Gaussian: {ssim_val:.6f}")
print(f"PSNR between spatial and frequency Gaussian: {psnr_val:.2f} dB")
print("\nThis confirms the Convolution Theorem: convolution in spatial domain ≈ multiplication in frequency domain.")

### 5.7 Quantitative Impact of Frequency Filtering on Membrane Structures

In [ ]:
# Measure how well each filter preserves membrane structures
# Using SSIM of filtered image vs original, and edge preservation ratio

freq_filter_results = []
test_idx_list = [0, 5, 10, 15, 20, 25, 29]

for idx in test_idx_list:
    img = train_images[idx].astype(np.float64)
    img_norm = img / 255.0

    for cutoff in [20, 40, 80]:
        all_filters = create_frequency_filters((512, 512), cutoff)

        for fname in ['Ideal LP', 'Butterworth LP', 'Gaussian LP']:
            result = apply_freq_filter(img, all_filters[fname]) / 255.0
            result = np.clip(result, 0, 1)

            s = ssim(img_norm, result, data_range=1.0)
            p = psnr(img_norm, result, data_range=1.0)

            freq_filter_results.append({
                'Filter': f'{fname} D₀={cutoff}',
                'Type': fname.split()[0],
                'Cutoff': cutoff,
                'Image': idx + 1,
                'SSIM': s,
                'PSNR': p
            })

freq_df = pd.DataFrame(freq_filter_results)
freq_summary = freq_df.groupby('Filter').agg(
    PSNR_Mean=('PSNR', 'mean'),
    SSIM_Mean=('SSIM', 'mean'),
).round(4).sort_values('SSIM_Mean', ascending=False)

print("=" * 65)
print("FREQUENCY-DOMAIN LP FILTER COMPARISON")
print("=" * 65)
print(f"{'Filter':<30} {'PSNR (dB)':>12} {'SSIM':>12}")
print("-" * 65)
for name, row in freq_summary.iterrows():
    print(f"{name:<30} {row['PSNR_Mean']:>12.2f} {row['SSIM_Mean']:>12.4f}")

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
colors_map = {'Ideal': '#E53935', 'Butterworth': '#43A047', 'Gaussian': '#1E88E5'}
bar_colors = [colors_map.get(freq_df[freq_df['Filter']==f]['Type'].iloc[0], 'gray') for f in freq_summary.index]

ax.barh(range(len(freq_summary)), freq_summary['SSIM_Mean'], color=bar_colors, alpha=0.85, edgecolor='gray')
ax.set_yticks(range(len(freq_summary)))
ax.set_yticklabels(freq_summary.index, fontsize=9)
ax.set_xlabel('SSIM (Higher = better structure preservation)')
ax.set_title('Frequency-Domain LP Filter — Structure Preservation', fontweight='bold')
ax.grid(axis='x', alpha=0.3)

from matplotlib.patches import Patch
legend_elems = [Patch(facecolor=c, label=l) for l, c in colors_map.items()]
ax.legend(handles=legend_elems, loc='lower right')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'freq_filter_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

### 5.8 Frequency-Domain Processing Summary

**Key findings:**

| Technique | Observation |
|---|---|
| **FFT Magnitude Spectrum** | EM images have strong low-frequency content with directional patterns reflecting tissue structure. The spectrum reveals periodic features not visible in the spatial domain. |
| **Ideal LP Filter** | Sharp cutoff causes visible ringing (Gibbs phenomenon). Not suitable for preprocessing. |
| **Butterworth LP Filter** | Smooth rolloff reduces ringing. Good balance between smoothing and artifact suppression. |
| **Gaussian LP Filter** | Smoothest result, no ringing. Mathematically equivalent to spatial Gaussian blur (convolution theorem). Best for preprocessing. |
| **High-Pass Filtering** | Extracts edges similarly to Sobel/Canny. Gaussian HP at D₀=40 captures membrane structures well. |
| **Band-Pass Filtering** | The 20–50 frequency band isolates membrane-scale structures. Useful for texture analysis. |

> **Connection to U-Net:** The encoder in U-Net progressively extracts features at different frequency scales — early layers capture high-frequency details (edges, textures) while deeper layers capture low-frequency features (overall cell shapes). This is analogous to multi-scale frequency analysis.

---

## 6. U-Net Implementation

### 6.1 Model Architecture

We implement the U-Net architecture following the original paper:
- **Encoder:** 4 blocks, each with 2×(Conv3×3 → BatchNorm → ReLU) + MaxPool2×2
- **Bottleneck:** 2×(Conv3×3 → BatchNorm → ReLU) with 1024 channels
- **Decoder:** 4 blocks, each with UpConv2×2 → Concatenate(skip) → 2×(Conv3×3 → BatchNorm → ReLU)
- **Output:** Conv1×1 → Sigmoid (binary segmentation)

We add BatchNorm (not in the original paper) for more stable training with our small dataset.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import random

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU available — training will be slower but will still work.")

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [ ]:
class DoubleConv(nn.Module):
    """Two consecutive Conv3x3 → BatchNorm → ReLU blocks."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.double_conv(x)


class UNet(nn.Module):
    """
    U-Net architecture for binary segmentation.
    Based on: Ronneberger et al. (2015) arXiv:1505.04597
    """
    def __init__(self, in_channels=1, out_channels=1, features=[64, 128, 256, 512]):
        super().__init__()
        self.encoder_blocks = nn.ModuleList()
        self.decoder_blocks = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.upconvs = nn.ModuleList()

        # Encoder path
        for feature in features:
            self.encoder_blocks.append(DoubleConv(in_channels, feature))
            in_channels = feature

        # Bottleneck
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        # Decoder path
        for feature in reversed(features):
            self.upconvs.append(nn.ConvTranspose2d(feature * 2, feature, kernel_size=2, stride=2))
            self.decoder_blocks.append(DoubleConv(feature * 2, feature))

        # Final 1x1 convolution
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []

        # Encoder
        for encoder_block in self.encoder_blocks:
            x = encoder_block(x)
            skip_connections.append(x)
            x = self.pool(x)

        # Bottleneck
        x = self.bottleneck(x)

        # Decoder
        skip_connections = skip_connections[::-1]  # Reverse order
        for idx in range(len(self.decoder_blocks)):
            x = self.upconvs[idx](x)

            # Handle size mismatch (crop skip connection if needed)
            skip = skip_connections[idx]
            if x.shape != skip.shape:
                x = TF.resize(x, size=skip.shape[2:])

            x = torch.cat((skip, x), dim=1)  # Concatenate along channel dimension
            x = self.decoder_blocks[idx](x)

        return torch.sigmoid(self.final_conv(x))


# Instantiate model and print summary
model = UNet(in_channels=1, out_channels=1).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"U-Net Model Summary:")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Device: {device}")

# Test with a dummy input
dummy = torch.randn(1, 1, 512, 512).to(device)
with torch.no_grad():
    output = model(dummy)
print(f"  Input shape:  {dummy.shape}")
print(f"  Output shape: {output.shape}")
print(f"  Output range: [{output.min():.4f}, {output.max():.4f}]")

### 6.2 Dataset & Data Pipeline

In [ ]:
class ISBIDataset(Dataset):
    """PyTorch Dataset for ISBI 2012 EM Segmentation data."""

    def __init__(self, images, labels, augment=False):
        self.images = images
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx].astype(np.float32) / 255.0
        label = self.labels[idx].astype(np.float32) / 255.0

        # Binarize label (ensure 0 or 1)
        label = (label > 0.5).astype(np.float32)

        # Convert to tensors
        image = torch.from_numpy(image).unsqueeze(0)  # (1, H, W)
        label = torch.from_numpy(label).unsqueeze(0)   # (1, H, W)

        if self.augment:
            # Random horizontal flip
            if random.random() > 0.5:
                image = TF.hflip(image)
                label = TF.hflip(label)

            # Random vertical flip
            if random.random() > 0.5:
                image = TF.vflip(image)
                label = TF.vflip(label)

            # Random rotation (0, 90, 180, or 270 degrees)
            angle = random.choice([0, 90, 180, 270])
            if angle > 0:
                image = TF.rotate(image, angle)
                label = TF.rotate(label, angle)

        return image, label


# Split: 24 training + 6 validation
val_indices = [0, 5, 10, 15, 20, 25]
train_indices = [i for i in range(30) if i not in val_indices]

train_imgs = train_images[train_indices]
train_lbls = train_labels[train_indices]
val_imgs = train_images[val_indices]
val_lbls = train_labels[val_indices]

train_dataset = ISBIDataset(train_imgs, train_lbls, augment=True)
val_dataset = ISBIDataset(val_imgs, val_lbls, augment=False)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0)

print(f"Training set:   {len(train_dataset)} images")
print(f"Validation set: {len(val_dataset)} images")
print(f"Train loader:   {len(train_loader)} batches (batch_size=2)")
print(f"Val loader:     {len(val_loader)} batches (batch_size=1)")

### 6.3 Training Loop

In [ ]:
def dice_coefficient(pred, target, smooth=1e-6):
    """Compute Dice coefficient."""
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    intersection = (pred_flat * target_flat).sum()
    return (2.0 * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)


def iou_score(pred, target, smooth=1e-6):
    """Compute IoU (Intersection over Union)."""
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    intersection = (pred_flat * target_flat).sum()
    union = pred_flat.sum() + target_flat.sum() - intersection
    return (intersection + smooth) / (union + smooth)


class DiceBCELoss(nn.Module):
    """Combined Dice + Binary Cross-Entropy Loss for better convergence."""
    def __init__(self, weight_bce=0.5, weight_dice=0.5):
        super().__init__()
        self.bce = nn.BCELoss()
        self.weight_bce = weight_bce
        self.weight_dice = weight_dice

    def forward(self, pred, target):
        bce_loss = self.bce(pred, target)
        dice_loss = 1 - dice_coefficient(pred, target)
        return self.weight_bce * bce_loss + self.weight_dice * dice_loss

In [ ]:
# Training configuration
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3

model = UNet(in_channels=1, out_channels=1).to(device)
criterion = DiceBCELoss(weight_bce=0.5, weight_dice=0.5)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

# Training history
history = {'train_loss': [], 'val_loss': [], 'val_dice': [], 'val_iou': [], 'val_acc': []}
best_val_dice = 0
best_model_state = None

print(f"Training U-Net for {NUM_EPOCHS} epochs...")
print(f"Optimizer: Adam (lr={LEARNING_RATE})")
print(f"Loss: DiceBCE (50% BCE + 50% Dice)")
print(f"Device: {device}")
print("=" * 70)

for epoch in range(NUM_EPOCHS):
    # ---- Training ----
    model.train()
    train_loss = 0
    for batch_imgs, batch_lbls in train_loader:
        batch_imgs = batch_imgs.to(device)
        batch_lbls = batch_lbls.to(device)

        optimizer.zero_grad()
        predictions = model(batch_imgs)
        loss = criterion(predictions, batch_lbls)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ---- Validation ----
    model.eval()
    val_loss = 0
    val_dice = 0
    val_iou = 0
    val_acc = 0

    with torch.no_grad():
        for batch_imgs, batch_lbls in val_loader:
            batch_imgs = batch_imgs.to(device)
            batch_lbls = batch_lbls.to(device)

            predictions = model(batch_imgs)
            loss = criterion(predictions, batch_lbls)
            val_loss += loss.item()

            # Binarize predictions
            preds_binary = (predictions > 0.5).float()
            val_dice += dice_coefficient(preds_binary, batch_lbls).item()
            val_iou += iou_score(preds_binary, batch_lbls).item()
            val_acc += (preds_binary == batch_lbls).float().mean().item()

    val_loss /= len(val_loader)
    val_dice /= len(val_loader)
    val_iou /= len(val_loader)
    val_acc /= len(val_loader)

    # Update scheduler
    scheduler.step(val_loss)

    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_dice'].append(val_dice)
    history['val_iou'].append(val_iou)
    history['val_acc'].append(val_acc)

    # Save best model
    if val_dice > best_val_dice:
        best_val_dice = val_dice
        best_model_state = model.state_dict().copy()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:>3}/{NUM_EPOCHS}] | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Dice: {val_dice:.4f} | "
              f"IoU: {val_iou:.4f} | "
              f"Acc: {val_acc:.4f}")

print("=" * 70)
print(f"Training complete! Best validation Dice: {best_val_dice:.4f}")

# Load best model
model.load_state_dict(best_model_state)

### 6.4 Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('U-Net Training Curves', fontsize=15, fontweight='bold')

epochs = range(1, NUM_EPOCHS + 1)

# Loss curves
axes[0].plot(epochs, history['train_loss'], 'b-', linewidth=2, label='Train Loss')
axes[0].plot(epochs, history['val_loss'], 'r-', linewidth=2, label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('(a) Training & Validation Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Dice & IoU curves
axes[1].plot(epochs, history['val_dice'], 'g-', linewidth=2, label='Dice')
axes[1].plot(epochs, history['val_iou'], 'm-', linewidth=2, label='IoU')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].set_title('(b) Validation Dice & IoU', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

# Pixel accuracy
axes[2].plot(epochs, history['val_acc'], 'orange', linewidth=2, label='Pixel Accuracy')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Accuracy')
axes[2].set_title('(c) Validation Pixel Accuracy', fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Final metrics — Dice: {history['val_dice'][-1]:.4f} | IoU: {history['val_iou'][-1]:.4f} | Acc: {history['val_acc'][-1]:.4f}")

## 7. Results & Evaluation

### 7.1 Segmentation Predictions — Visual Results

In [ ]:
# Visualize predictions on validation set
model.eval()

fig, axes = plt.subplots(3, 6, figsize=(24, 12))
fig.suptitle('U-Net Segmentation Results on Validation Set', fontsize=15, fontweight='bold', y=1.01)

for i, idx in enumerate(val_indices):
    img = train_images[idx].astype(np.float32) / 255.0
    lbl = train_labels[idx].astype(np.float32) / 255.0
    lbl_binary = (lbl > 0.5).astype(np.float32)

    img_tensor = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(img_tensor).cpu().numpy()[0, 0]

    pred_binary = (pred > 0.5).astype(np.float32)

    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].set_title(f'Frame {idx+1}', fontweight='bold')
    axes[0, i].axis('off')

    axes[1, i].imshow(lbl_binary, cmap='gray')
    axes[1, i].set_title('Ground Truth', fontweight='bold')
    axes[1, i].axis('off')

    axes[2, i].imshow(pred_binary, cmap='gray')
    d = dice_coefficient(torch.tensor(pred_binary), torch.tensor(lbl_binary)).item()
    axes[2, i].set_title(f'Prediction (Dice={d:.3f})', fontweight='bold')
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('Input', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Ground Truth', fontsize=12, fontweight='bold')
axes[2, 0].set_ylabel('U-Net Output', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'segmentation_results.png'), dpi=150, bbox_inches='tight')
plt.show()

### 7.2 Quantitative Metrics

In [ ]:
# Compute metrics on entire validation set
all_dice, all_iou, all_acc, all_prec, all_rec = [], [], [], [], []

model.eval()
for idx in val_indices:
    img = train_images[idx].astype(np.float32) / 255.0
    lbl = train_labels[idx].astype(np.float32) / 255.0
    lbl_binary = (lbl > 0.5).astype(np.float32)

    img_tensor = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(img_tensor).cpu().numpy()[0, 0]
    pred_binary = (pred > 0.5).astype(np.float32)

    # Metrics
    d = dice_coefficient(torch.tensor(pred_binary), torch.tensor(lbl_binary)).item()
    iou = iou_score(torch.tensor(pred_binary), torch.tensor(lbl_binary)).item()
    acc = (pred_binary == lbl_binary).mean()

    # Precision & Recall
    tp = ((pred_binary == 1) & (lbl_binary == 1)).sum()
    fp = ((pred_binary == 1) & (lbl_binary == 0)).sum()
    fn = ((pred_binary == 0) & (lbl_binary == 1)).sum()
    prec = tp / (tp + fp + 1e-8)
    rec = tp / (tp + fn + 1e-8)

    all_dice.append(d)
    all_iou.append(iou)
    all_acc.append(acc)
    all_prec.append(prec)
    all_rec.append(rec)

print("=" * 60)
print("U-NET VALIDATION METRICS")
print("=" * 60)
print(f"{'Metric':<25} {'Mean':>10} {'Std':>10}")
print("-" * 60)
print(f"{'Dice Coefficient':<25} {np.mean(all_dice):>10.4f} {np.std(all_dice):>10.4f}")
print(f"{'IoU (Jaccard)':<25} {np.mean(all_iou):>10.4f} {np.std(all_iou):>10.4f}")
print(f"{'Pixel Accuracy':<25} {np.mean(all_acc):>10.4f} {np.std(all_acc):>10.4f}")
print(f"{'Precision':<25} {np.mean(all_prec):>10.4f} {np.std(all_prec):>10.4f}")
print(f"{'Recall':<25} {np.mean(all_rec):>10.4f} {np.std(all_rec):>10.4f}")

### 7.3 Comparison with Original Paper

In [ ]:
# Create comparison table
our_dice = np.mean(all_dice)
our_iou = np.mean(all_iou)
our_acc = np.mean(all_acc)

print("=" * 70)
print("COMPARISON WITH ORIGINAL U-NET PAPER")
print("=" * 70)
print(f"{'Metric':<25} {'Original Paper':>18} {'Our Implementation':>18}")
print("-" * 70)
print(f"{'Warping Error':<25} {'0.000353':>18} {'N/A (needs ISBI submission)':>18}")
print(f"{'Rand Error':<25} {'0.0382':>18} {'N/A (needs ISBI submission)':>18}")
print(f"{'Dice Coefficient':<25} {'~0.92 (PhC-U373)':>18} {our_dice:>18.4f}")
print(f"{'IoU':<25} {'0.9203 (PhC-U373)':>18} {our_iou:>18.4f}")
print(f"{'Pixel Accuracy':<25} {'—':>18} {our_acc:>18.4f}")
print()
print("NOTES ON DIFFERENCES:")
print("1. The paper's IoU=0.9203 was on PhC-U373 (different dataset), not ISBI 2012 EM.")
print("2. The paper used 7-rotation test-time averaging — we did not implement this.")
print("3. The paper used Caffe + NVidia Titan GPU + 10 hours training.")
print(f"4. We used PyTorch + {'GPU (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU'} + {NUM_EPOCHS} epochs.")
print("5. The paper used elastic deformations for augmentation — we used flips + rotations.")
print("6. We added BatchNorm (not in original) for more stable training with small data.")

## 8. Discussion & Limitations

### Key Observations

1. **Spatial-domain processing** (Section 4) demonstrated that classical edge detection methods can capture membrane structures, but they lack the contextual understanding that a deep network provides. Canny edge detection with σ=1.0 was the best classical method, but still significantly weaker than the learned U-Net features.

2. **Frequency-domain processing** (Section 5) confirmed the convolution theorem — spatial Gaussian blur and frequency-domain Gaussian LP filtering produce equivalent results. The frequency analysis also revealed that membrane structures occupy a specific frequency band (roughly 20–50 cycles/image), which explains why the U-Net's multi-scale architecture is effective.

3. **U-Net segmentation** achieved promising results despite limited training data. The encoder-decoder architecture with skip connections effectively balances context and localization.

### Limitations

1. **Small dataset:** Only 30 images (24 train / 6 val) — high risk of overfitting.
2. **No test-time augmentation:** The paper averaged 7 rotated predictions; we only used single forward passes.
3. **Simplified augmentation:** We used flips and rotations but NOT elastic deformations (the paper's key augmentation technique).
4. **Different framework:** PyTorch vs. original Caffe implementation may produce slightly different results.
5. **No weight map:** The paper's custom weight map for touching cell boundaries was not implemented.
6. **Limited training time:** The paper trained for 10 hours on a Titan GPU; our training was shorter.

### What Would Improve Results

- Implementing elastic deformation augmentation
- Adding the paper's weight map (Equation 2) to the loss function
- Training for more epochs with a larger GPU
- Test-time augmentation with 7 rotations
- Using the original unpadded convolutions instead of padded ones

---

## 9. References

1. **Ronneberger, O., Fischer, P., & Brox, T. (2015).** U-Net: Convolutional Networks for Biomedical Image Segmentation. *arXiv:1505.04597*. [https://arxiv.org/abs/1505.04597](https://arxiv.org/abs/1505.04597)

2. **ISBI 2012 EM Segmentation Challenge.** [http://brainiac2.mit.edu/isbi_challenge/](http://brainiac2.mit.edu/isbi_challenge/)

3. **Ciresan, D.C., Gambardella, L.M., Giusti, A., & Schmidhuber, J. (2012).** Deep neural networks segment neuronal membranes in electron microscopy images. *NIPS*, pp. 2852–2860.

4. **Long, J., Shelhamer, E., & Darrell, T. (2015).** Fully convolutional networks for semantic segmentation. *CVPR*.

### Libraries Used
- Python 3.x, NumPy, Pandas, Matplotlib, Seaborn
- Scikit-image, OpenCV, SciPy
- PyTorch
- Scikit-learn

---

*Notebook prepared by Özgür Efe Zurnacı (221402001) for BME 562/506 Medical Image Processing, Spring 2025–2026.*